
# Práctica 2 — Análisis estadístico de señales
## Parte 1: señal ECG del archivo `signals.mat`

**Objetivo:** analizar una señal ECG en el dominio del tiempo usando medidas estadísticas sencillas.

En este notebook voy a trabajar paso a paso, usando un código sencillo y comentarios cortos.  
Primero cargo la señal, luego comparo la señal original con la filtrada, calculo estadísticas de ciclos cardiacos y finalmente reviso la estacionariedad.



## 1. Librerías que vamos a utilizar

- `numpy`: para hacer operaciones matemáticas.
- `matplotlib`: para hacer gráficas.
- `scipy.io`: para abrir el archivo `.mat`.
- `scipy.signal`: para encontrar los picos R del ECG.
- `scipy.stats`: para las pruebas estadísticas.
- `statsmodels`: para la prueba de Dickey-Fuller.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings

from scipy.io import loadmat
from scipy.signal import find_peaks
from scipy.stats import shapiro, levene, ttest_ind, mannwhitneyu
from statsmodels.tsa.stattools import adfuller

# Algunas librerías (como statsmodels) muestran advertencias internas que no son errores
# y no afectan los resultados; las ocultamos para que la salida sea más limpia y fácil de leer.
warnings.filterwarnings("ignore")




## 2. Subir el archivo `signals.mat`

Al ejecutar esta celda, Colab muestra un botón para seleccionar el archivo desde el computador.


In [ ]:

from google.colab import files

archivos = files.upload()



## 3. Cargar el archivo y revisar qué contiene


In [ ]:

datos = loadmat("signals.mat")

# Muestro los nombres de las variables guardadas en el archivo
print(datos.keys())



En el archivo aparecen las señales de ECG y EMG.  
Para esta parte de la práctica vamos a usar:

- `ECG_asRecording`: ECG tal como fue registrado.
- `ECG_filtered`: ECG después del filtrado.
- `Fs`: frecuencia de muestreo.


In [ ]:

Fs = int(np.squeeze(datos["Fs"]))

ecg_original = np.squeeze(datos["ECG_asRecording"])
ecg_filtrada = np.squeeze(datos["ECG_filtered"])

print("Frecuencia de muestreo:", Fs, "Hz")
print("Muestras ECG original:", len(ecg_original))
print("Muestras ECG filtrada:", len(ecg_filtrada))



## 4. Función para calcular el RMS

El RMS o **valor cuadrático medio** se calcula como:

\[
RMS = \sqrt{\frac{1}{N}\sum_{i=1}^{N}x(i)^2}
\]

Voy a crear una función para poder utilizarla varias veces.


In [ ]:

def calcular_rms(senal):
    rms = np.sqrt(np.mean(senal**2))
    return rms



## 5. Duración de la señal y vector de tiempo

La duración se obtiene dividiendo el número de muestras entre la frecuencia de muestreo:

\[
duración = \frac{N}{Fs}
\]


In [ ]:

numero_muestras = len(ecg_original)
duracion = numero_muestras / Fs

tiempo = np.arange(numero_muestras) / Fs

print("Duración de la señal:", duracion, "segundos")



### Interpretación

Con este archivo, el ECG tiene **30 segundos de duración** porque contiene 30720 muestras y fue adquirido a 1024 Hz.



## 6. Comparación entre la señal original y la señal filtrada


In [ ]:

plt.figure(figsize=(15, 6))

plt.plot(tiempo, ecg_original, label="ECG original", alpha=0.7)
plt.plot(tiempo, ecg_filtrada, label="ECG filtrada", alpha=0.8)

plt.xlabel("Tiempo (s)")
plt.ylabel("Amplitud")
plt.title("Comparación ECG original y filtrada")
plt.legend()
plt.grid()
plt.show()



### Análisis de la gráfica

Se observa que la señal original tiene un desplazamiento grande hacia valores positivos, mientras que la señal filtrada queda aproximadamente centrada alrededor de cero.

Además, en la señal filtrada se distinguen mejor los complejos del ECG y se reduce parte del ruido.

Por lo tanto, el filtro parece eliminar componentes muy lentas o continuas de la señal, como el desplazamiento de la línea base, y también reducir ruido no deseado. Su comportamiento es parecido al de un **filtro pasa banda**, porque busca conservar principalmente las frecuencias útiles del ECG.

**Importante:** con el archivo entregado podemos observar el efecto del filtro, pero no conocemos sus frecuencias de corte exactas.


### Comprobando el filtro con el espectro de frecuencia

La comparación anterior fue visual. Para tener una evidencia más objetiva de qué frecuencias eliminó el filtro, calculamos el espectro de frecuencia (usando la Transformada de Fourier) de la señal original y de la filtrada.


In [ ]:
# Le quitamos la media antes de la FFT para no tener un pico gigante en 0 Hz
fft_original = np.abs(np.fft.rfft(ecg_original - np.mean(ecg_original)))
fft_filtrada = np.abs(np.fft.rfft(ecg_filtrada - np.mean(ecg_filtrada)))
frecuencias = np.fft.rfftfreq(numero_muestras, d=1/Fs)

plt.figure(figsize=(12, 5))
plt.plot(frecuencias, fft_original, label="ECG original", alpha=0.7)
plt.plot(frecuencias, fft_filtrada, label="ECG filtrada", alpha=0.8)
plt.xlim(0, 50)
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("Magnitud")
plt.title("Espectro de frecuencia: original vs filtrada")
plt.legend()
plt.grid()
plt.show()

### Interpretación del espectro

En el espectro se observa que la señal filtrada tiene mucha menos magnitud cerca de 0 Hz que la señal original. Esto confirma, con datos y no solo de forma visual, que el filtro elimina la componente continua y las frecuencias muy bajas (como el desplazamiento de línea base).

También se nota una pequeña reducción en frecuencias altas (por encima de 40 Hz aproximadamente), lo que sugiere que el filtro también atenúa ruido de alta frecuencia.

En la banda intermedia, donde están las frecuencias de interés del ECG (aproximadamente 0.5 Hz a 40 Hz), la magnitud se conserva casi igual entre las dos señales. Esto confirma que el filtro se comporta como un **filtro pasa banda**, tal como se había planteado antes.


## 7. Encontrar los picos R para separar los ciclos cardiacos

Para poder extraer ciclos cardiacos voy a localizar automáticamente los picos R.

Uso dos condiciones sencillas:

- Los picos deben estar separados al menos 0.5 segundos.
- Deben ser suficientemente grandes con respecto a la variación de la señal.


In [ ]:

distancia_minima = int(0.5 * Fs)
prominencia = 3 * np.std(ecg_filtrada)

picos_R, propiedades = find_peaks(
    ecg_filtrada,
    distance=distancia_minima,
    prominence=prominencia
)

print("Número de picos R encontrados:", len(picos_R))


In [ ]:

plt.figure(figsize=(15, 5))

plt.plot(tiempo, ecg_filtrada, label="ECG filtrada")
plt.plot(tiempo[picos_R], ecg_filtrada[picos_R], "ro", label="Picos R")

plt.xlim(0, 10)
plt.xlabel("Tiempo (s)")
plt.ylabel("Amplitud")
plt.title("Detección de picos R")
plt.legend()
plt.grid()
plt.show()



Si los puntos rojos están ubicados sobre los picos altos del ECG, la detección está funcionando correctamente.



## 8. Seleccionar un ciclo de la señal original

Voy a tomar como ciclo cardiaco el intervalo comprendido entre un pico R y el siguiente.

Usaré el mismo intervalo después para comparar la señal original y la filtrada.


In [ ]:

# Elijo el ciclo número 5 para evitar el inicio del registro
numero_ciclo = 5

inicio = picos_R[numero_ciclo]
fin = picos_R[numero_ciclo + 1]

ciclo_original = ecg_original[inicio:fin]
tiempo_ciclo = np.arange(len(ciclo_original)) / Fs

plt.figure(figsize=(10, 4))
plt.plot(tiempo_ciclo, ciclo_original)

plt.xlabel("Tiempo desde el inicio del ciclo (s)")
plt.ylabel("Amplitud")
plt.title("Un ciclo cardiaco - ECG original")
plt.grid()
plt.show()



## 9. Estadísticas del ciclo original


In [ ]:

media_original = np.mean(ciclo_original)
rms_original = calcular_rms(ciclo_original)
varianza_original = np.var(ciclo_original)
desviacion_original = np.std(ciclo_original)

print("Media:", media_original)
print("RMS:", rms_original)
print("Varianza:", varianza_original)
print("Desviación estándar:", desviacion_original)



### Interpretación

La media del ECG original es alta porque la señal tiene un desplazamiento de línea base importante.

El RMS también resulta alto porque depende de la magnitud total de los valores de la señal.

La varianza y la desviación estándar muestran cuánto cambian los valores dentro del ciclo cardiaco.



## 10. Repetir el análisis con la señal filtrada

Voy a utilizar exactamente el mismo intervalo del ciclo anterior para que la comparación sea justa.


In [ ]:

ciclo_filtrado = ecg_filtrada[inicio:fin]

plt.figure(figsize=(10, 4))
plt.plot(tiempo_ciclo, ciclo_filtrado)

plt.xlabel("Tiempo desde el inicio del ciclo (s)")
plt.ylabel("Amplitud")
plt.title("Un ciclo cardiaco - ECG filtrada")
plt.grid()
plt.show()


In [ ]:

media_filtrada = np.mean(ciclo_filtrado)
rms_filtrada = calcular_rms(ciclo_filtrado)
varianza_filtrada = np.var(ciclo_filtrado)
desviacion_filtrada = np.std(ciclo_filtrado)

print("Media:", media_filtrada)
print("RMS:", rms_filtrada)
print("Varianza:", varianza_filtrada)
print("Desviación estándar:", desviacion_filtrada)



### Comparación

Después del filtrado, la media queda mucho más cerca de cero. Esto indica que el filtro eliminó gran parte del desplazamiento o componente continua de la señal.

El RMS también disminuye bastante porque ya no está incluido ese desplazamiento grande.

La varianza y la desviación estándar pueden mantenerse en un rango parecido porque todavía se conserva la forma variable del ciclo cardiaco.



## 11. Extraer 15 ciclos de la señal filtrada


In [ ]:
# Verificamos que hay suficientes picos R para sacar 15 ciclos
# (si el registro fuera más corto o el filtro cambiara, esto avisaría del problema)
print("Picos R disponibles:", len(picos_R))
assert len(picos_R) >= 17, "No hay suficientes picos R para extraer 15 ciclos"


ciclos = []

# Empiezo desde el segundo pico para evitar el borde inicial
for i in range(1, 16):
    inicio_ciclo = picos_R[i]
    fin_ciclo = picos_R[i + 1]

    ciclo = ecg_filtrada[inicio_ciclo:fin_ciclo]
    ciclos.append(ciclo)

print("Cantidad de ciclos guardados:", len(ciclos))



## 12. Graficar los 15 ciclos


In [ ]:

fig, ejes = plt.subplots(5, 3, figsize=(15, 15))

for i in range(15):
    fila = i // 3
    columna = i % 3

    tiempo_local = np.arange(len(ciclos[i])) / Fs

    ejes[fila, columna].plot(tiempo_local, ciclos[i])
    ejes[fila, columna].set_title("Ciclo " + str(i + 1))
    ejes[fila, columna].set_xlabel("Tiempo (s)")
    ejes[fila, columna].set_ylabel("Amplitud")
    ejes[fila, columna].grid()

plt.tight_layout()
plt.show()



## 13. Media y varianza de los 15 ciclos


In [ ]:

medias = []
varianzas = []

for ciclo in ciclos:
    medias.append(np.mean(ciclo))
    varianzas.append(np.var(ciclo))

tabla_ciclos = pd.DataFrame({
    "Ciclo": np.arange(1, 16),
    "Media": medias,
    "Varianza": varianzas
})

tabla_ciclos


In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(tabla_ciclos["Ciclo"], tabla_ciclos["Media"], marker="o")
plt.xlabel("Ciclo")
plt.ylabel("Media")
plt.title("Media de cada ciclo")
plt.grid()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(tabla_ciclos["Ciclo"], tabla_ciclos["Varianza"], marker="o")
plt.xlabel("Ciclo")
plt.ylabel("Varianza")
plt.title("Varianza de cada ciclo")
plt.grid()
plt.show()



### Análisis de estacionariedad a partir de los ciclos

Las medias de los ciclos permanecen relativamente cerca de cero, pero no son exactamente iguales.

Las varianzas presentan cambios más evidentes entre los ciclos. Esto significa que la señal no mantiene exactamente las mismas características estadísticas durante todo el tiempo.

Por eso, desde una observación estricta, no se puede afirmar que el ECG sea perfectamente estacionario. Sin embargo, durante intervalos cortos puede comportarse de forma aproximadamente estacionaria.



## 14. Comparar dos pares de ciclos con pruebas estadísticas

La guía pide revisar primero:

1. **Normalidad**.
2. **Independencia**: para esta práctica se asume.
3. **Homocedasticidad** mediante Levene.

Si no se cumplen los requisitos de la prueba t, se utiliza la prueba **U de Mann-Whitney**.

Usaremos un nivel de significancia:

\[
\alpha = 0.05
\]

**Nota:** es normal que la prueba de Shapiro casi siempre rechace la normalidad en un ciclo cardiaco completo. Esto pasa porque el ciclo incluye el complejo QRS, que es un pico muy pronunciado y hace que la forma de la señal no se parezca a una campana de Gauss. Por eso es de esperar que en varias comparaciones se termine usando la prueba U de Mann-Whitney.


In [ ]:

def comparar_ciclos(ciclo1, ciclo2, nombre1, nombre2):
    alpha = 0.05

    # 1. Prueba de normalidad
    normalidad1 = shapiro(ciclo1)
    normalidad2 = shapiro(ciclo2)

    print("Comparación:", nombre1, "vs", nombre2)
    print()
    print("Normalidad", nombre1, "- p =", normalidad1.pvalue)
    print("Normalidad", nombre2, "- p =", normalidad2.pvalue)

    # 2. Prueba de Levene
    levene_resultado = levene(ciclo1, ciclo2)

    print("Levene - p =", levene_resultado.pvalue)
    print()

    normal1 = normalidad1.pvalue > alpha
    normal2 = normalidad2.pvalue > alpha
    misma_varianza = levene_resultado.pvalue > alpha

    # 3. Elegir la prueba
    if normal1 and normal2 and misma_varianza:
        resultado = ttest_ind(ciclo1, ciclo2, equal_var=True)

        print("Se cumplen los supuestos.")
        print("Se utiliza prueba t de Student.")
        print("Estadístico =", resultado.statistic)
        print("p-valor =", resultado.pvalue)

    else:
        resultado = mannwhitneyu(ciclo1, ciclo2, alternative="two-sided")

        print("No se cumplen todos los supuestos de la prueba t.")
        print("Se utiliza U de Mann-Whitney.")
        print("Estadístico =", resultado.statistic)
        print("p-valor =", resultado.pvalue)

    # 4. Decisión
    if resultado.pvalue < alpha:
        print("Decisión: se rechaza H0.")
        print("Hay diferencia estadísticamente significativa entre los ciclos.")
    else:
        print("Decisión: no se rechaza H0.")
        print("No se encontró diferencia estadísticamente significativa entre los ciclos.")



### Primera comparación: ciclo 1 vs ciclo 5


In [ ]:

comparar_ciclos(
    ciclos[0],
    ciclos[4],
    "Ciclo 1",
    "Ciclo 5"
)



### Segunda comparación: ciclo 6 vs ciclo 13


In [ ]:

comparar_ciclos(
    ciclos[5],
    ciclos[12],
    "Ciclo 6",
    "Ciclo 13"
)


### Hipótesis usadas

**H0:** no existen diferencias estadísticamente significativas entre los dos ciclos comparados.

**H1:** existen diferencias estadísticamente significativas entre los dos ciclos comparados.

En nuestros resultados, la comparación Ciclo 1 vs Ciclo 5 dio un p-valor mayor a 0.05 (no se rechaza H0, los ciclos no son estadísticamente distintos), mientras que la comparación Ciclo 6 vs Ciclo 13 dio un p-valor menor a 0.05 (se rechaza H0, sí hay diferencia estadística).

Esto muestra que el comportamiento no es siempre igual: algunos pares de ciclos son estadísticamente iguales y otros no. Esa inconsistencia es justamente evidencia de que la señal no es perfectamente estacionaria, aunque tampoco cambia de forma completamente aleatoria.

Sin embargo, comparar solamente dos pares de ciclos no es suficiente para describir por completo la estacionariedad de toda la señal. Por eso también se utiliza Dickey-Fuller.



## 15. Prueba de Dickey-Fuller

La prueba Dickey-Fuller aumentada evalúa la presencia de una raíz unitaria.

De forma sencilla:

- **H0:** la señal tiene raíz unitaria y se considera no estacionaria.
- **H1:** la señal no tiene raíz unitaria y presenta evidencia a favor de estacionariedad.

Si `p < 0.05`, rechazamos H0.


In [ ]:

resultado_adf = adfuller(ecg_filtrada)

print("Estadístico ADF:", resultado_adf[0])
print("p-valor:", resultado_adf[1])

if resultado_adf[1] < 0.05:
    print("Se rechaza H0.")
    print("La prueba ADF encuentra evidencia a favor de estacionariedad.")
else:
    print("No se rechaza H0.")
    print("La prueba ADF no permite considerar la señal estacionaria.")



### Interpretación conjunta

La prueba de Dickey-Fuller puede indicar que la señal filtrada tiene evidencia estadística a favor de estacionariedad en el sentido evaluado por esta prueba.

Sin embargo, al comparar ciclos se observan cambios en la media, la varianza y, en algunos casos, diferencias estadísticas entre ciclos.

Esto no es necesariamente una contradicción: Dickey-Fuller evalúa principalmente la existencia de una raíz unitaria, mientras que una bioseñal puede presentar otros cambios temporales. Por eso, una conclusión razonable es que el ECG filtrado puede comportarse como **aproximadamente estacionario en ciertos intervalos**, pero no debe considerarse perfectamente estacionario en sentido estricto.



# Conclusiones

1. El registro ECG analizado tiene una duración de 30 segundos y fue adquirido a 1024 Hz.
2. La señal original presenta un desplazamiento importante de la línea base.
3. El filtrado centra la señal cerca de cero y permite observar con mayor claridad la actividad cardiaca.
4. La media, RMS, varianza y desviación estándar permiten describir cuantitativamente los ciclos cardiacos.
5. Los ciclos no son exactamente iguales entre sí y sus estadísticas cambian ligeramente con el tiempo.
6. Las pruebas estadísticas permiten estudiar estas diferencias de una forma objetiva.
7. La estacionariedad del ECG debe interpretarse con cuidado, ya que las bioseñales pueden comportarse de forma aproximadamente estacionaria durante intervalos cortos, aunque no lo sean estrictamente.



# Referencias

- Guía de laboratorio: **Práctica 2 — Análisis estadístico de señales, Bioseñales y Sistemas**.
- SciPy documentation — Statistical functions and signal processing.
- Statsmodels documentation — Augmented Dickey-Fuller test.
